# C.7 Reward 怎麼改變輸出機率？


模型只在兩個答案間選擇：動作0代表答3，動作1代表答4。起初兩者分數都是0，機率各0.5；這次選動作1，驗證得到reward=1。若平常參考獎勵baseline=0.5，這次比參考好0.5，就希望提高答4的機率。前置是[可驗證獎勵](https://birdhackor.github.io/tiny-perceptron-vlm/C.6.html)、[機率與log](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.4)和[梯度更新](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.6)。本節只追蹤兩個可調分數的一次更新。

輸出分布稱policy（策略），這裡就是「答3或答4各有多大機率」。reward減baseline叫advantage，表示這次比參考好或差多少。對選到的動作，令誤差為 `-(reward-baseline)*logp(action)`。當advantage為正，降低這個誤差就會提高所選動作的log機率；當它為負，方向相反。這種用回饋調整動作機率的方法叫policy gradient（策略梯度）。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

logits = torch.tensor([0.0, 0.0], requires_grad=True)
action = 1
reward, baseline = 1.0, 0.5
before = logits.detach().softmax(0)
loss = -(reward - baseline) * logits.log_softmax(0)[action]
loss.backward()
with torch.no_grad():
    updated = logits - 0.1 * logits.grad
    after = updated.softmax(0)
print("梯度", logits.grad.tolist())
print("新分數", updated.tolist())
print("前機率", before.tolist())
print("後機率", [round(p, 6) for p in after.tolist()])

`detach()`取出數值相同、但不再追蹤梯度的tensor；`before`用這些分數算出更新前機率，只供前後比較，不參與後面的梯度計算。`log_softmax(0)`在兩個候選間算log機率，`[action]`只取這次選的動作1。reward與baseline是固定Python數字，不對驗證器反向傳播。`backward`得到梯度 `[0.25,-0.25]`；用步幅0.1減掉梯度，分數約成 `[-0.025,0.025]`。機率從 `[0.5,0.5]`變成 `[0.487503,0.512497]`，因此答4稍微更可能，不是一次就變100%。

下圖的橫箭頭追蹤分數更新，兩條向下箭頭各自將更新前後的分數轉成機率；清單順序固定為答3、答4。答4分數增加0.025後，機率只增約0.012497，不是把分數變動直接當成機率變動。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/architecture_policy_update.svg")))

這個程式直接調整兩個分數，沒有訓練語言模型；action也人工固定，沒有真的採樣。一般模型需先依policy抽出回答，得到回饋，再透過這類誤差信號更新產生分數的權重。baseline是回饋的參考值，合適的估計可減少採樣造成的波動；它不是正確答案，也不是把reward硬改成較高分。單次樣本可能不足以代表整體行為，所以不能用這次機率上升宣告訓練成功。

若回饋來自可檢查的真值或測試，這類訓練常稱RLVR（Reinforcement Learning with Verifiable Rewards，可驗證獎勵的強化學習）。它與[測試時多花算力](https://birdhackor.github.io/tiny-perceptron-vlm/C.5.html)不同：本例確實改了分數，一般訓練會改權重。完成後仍應在[獨立測試題](https://birdhackor.github.io/tiny-perceptron-vlm/5.10.html)檢查成功率、生成長度與評分漏洞，避免只學會迎合驗證器。

練習只把reward改成0，baseline仍0.5。先預測advantage變負、梯度改為 `[-0.25,0.25]`，更新後動作1機率下降到約0.487503，再執行核對。請用「本次比參考差」說明方向，而不是把所有非負reward都理解成應該提高機率。
